# In-Context Memory: The LLM's "Working Memory"

In-context memory is the most fundamental way an agent "remembers." It involves keeping a record of the conversation history and passing it back to the model with every new request.

### Core Concept:
- Every time you send a new message, you also send the *previous* messages.
- **Pros**: Extremely fast, no database required, very accurate.
- **Cons**: Limited by the context window; as the conversation grows, it becomes more expensive (tokens).

---

## 1. Manual History Management (The Basic Loop)
The simplest way to implement memory is by managing a Python list of message objects. We manually append user queries and AI responses to this list.

In [6]:
import os
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.schema import HumanMessage, AIMessage, SystemMessage

load_dotenv()
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash")

# Initialize our "Context Buffer"
history = [
    SystemMessage(content="You are a helpful assistant who remembers names.")
]

def chat_with_manual_memory(user_input):
    # Append query
    history.append(HumanMessage(content=user_input))
    
    # Invoke model with the entire history
    response = llm.invoke(history)
    
    # Record response
    history.append(response)
    
    return response.content

print(chat_with_manual_memory("Hi, my name is Bala."))
print(chat_with_manual_memory("What is my name?"))

Hi Bala! It's nice to meet you. I'll remember that.
Your name is Bala.


## 2. Context Windowing (Sliding Window)
As conversations get longer, the history consumes more tokens and can eventually exceed the model's limits. A **Sliding Window** strategy keeps only the most recent *N* messages.

In [7]:
def chat_with_window_memory(user_input, window_size=5):
    # Add user message
    history.append(HumanMessage(content=user_input))
    
    # Trim history to the last 'window_size' messages
    # Note: We usually want to KEEP the System Message at the start
    system_msg = history[0]
    recent_history = [system_msg] + history[-(window_size-1):] if len(history) > window_size else history
    
    response = llm.invoke(recent_history)
    history.append(response)
    
    return response.content

print("Window memory implementation ready.")

Window memory implementation ready.


## 3. Conversation Summarization
Instead of deleting old messages, we can have the LLM **summarize** them. This compresses the context while retaining the "gist" of earlier turns.

In [8]:
def summarize_history(messages):
    summary_prompt = "Summarize the following conversation history into a concise paragraph that captures key facts and user preferences:"
    res = llm.invoke([HumanMessage(content=summary_prompt + str(messages))])
    return res.content

def chat_with_summarization(user_input, threshold=10):
    global history
    
    if len(history) > threshold:
        print("--- History threshold reached. Summarizing... ---")
        summary = summarize_history(history)
        # Replace existing history with the summary
        history = [
            SystemMessage(content=f"You are a helpful assistant. Here is a summary of the past: {summary}")
        ]
    
    history.append(HumanMessage(content=user_input))
    response = llm.invoke(history)
    history.append(response)
    return response.content

print("Summarization logic ready.")

Summarization logic ready.


## 4. Managed Memory Components (Professional Patterns)
Modern frameworks like LangChain provide `ConversationSummaryBufferMemory` which automates both windowing and summarization behind the scenes.

In [9]:
from langchain.memory import ConversationSummaryBufferMemory
from langchain.chains import ConversationChain

# This component automatically summarizes history once it hits a token limit (max_token_limit)
memory = ConversationSummaryBufferMemory(llm=llm, max_token_limit=100)

conversation = ConversationChain(
    llm=llm, 
    memory=memory,
    verbose=True
)

print("LangChain Managed Memory initialized.")

LangChain Managed Memory initialized.


## Summary & Tips
1. **Always Monitor Tokens**: Long histories are expensive and can degrade model performance (the 'loss in the middle' problem).
2. **The Prompt is King**: Use the `SystemMessage` to inject high-priority context that the model must NEVER forget.
3. **Hybrid Approach**: For production, we often use In-Context memory for the last 5-10 turns and **Long-Term Memory** (Vector Stores) for things that happened weeks ago.